# Day 4 · Year-over-Year, Decomposition & More Risk Drivers

*Work log: a cleaned version of my working notebook. The polished analysis lives in [`../01`–`05`](../).*

**Goals:** confirm the 2016 slowdown with year-over-year comparisons, decompose it into its drivers, and profile three more application-time variables (DTI, employment length, purpose) with pandas.

**Key results:** funded volume fell YoY for **10 straight months** (Jul-2016 to Apr-2017). H2-2016 vs. H2-2015: funded **−21.9%**, driven by the **approval rate (−45.4%)** while applications **rose 53.9%**. No single application variable separates risk by more than **1.9×**, against **7.8×** for grade.

## 1. Setup

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA = (Path.cwd().parents[1] / "Data").resolve()   # project_root/Data
con = duckdb.connect()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

In [2]:
# Rebuild the analysis views (DuckDB views are in-memory and reset every session)

# loans: every accepted loan
con.sql(f"""
CREATE OR REPLACE VIEW loans AS
SELECT *
FROM '{DATA}/accepted.parquet'
""")

# loans_clean: drop the 33 file-footer rows and parse the text issue date ('Dec-2015') into a real date
con.sql("""
CREATE OR REPLACE VIEW loans_clean AS
SELECT *,
       strptime(issue_d, '%b-%Y') AS issue_date
FROM loans
WHERE loan_status IS NOT NULL
""")

# cohort: 36-month loans issued 2014-2015, all matured by Dec-2018 so outcomes are final
con.sql("""
CREATE OR REPLACE VIEW cohort AS
SELECT *
FROM loans_clean
WHERE issue_date BETWEEN '2014-01-01' AND '2015-12-31'
  AND TRIM(term) = '36 months'
""")

# rejected: declined applications
con.sql(f"""
CREATE OR REPLACE VIEW rejected AS
SELECT *
FROM '{DATA}/rejected.parquet'
""")

con.sql("SHOW TABLES").df()

,name
0,cohort
1,loans
2,loans_clean
3,rejected


## 2. Year-over-year change
`LAG(x, 12)` pulls the value from 12 rows earlier, i.e. the same month last year (valid because no month is missing).

In [3]:
con.sql("""
WITH monthly AS (
    SELECT DATE_TRUNC('month', issue_date) AS issue_month,
           SUM(funded_amnt) / 1000000      AS amount_mm     -- keep full precision inside the CTE
    FROM loans_clean
    WHERE issue_date >= '2015-01-01' AND issue_date < '2018-01-01'
    GROUP BY issue_month
)
SELECT issue_month,
       ROUND(amount_mm, 1)                                                      AS amount_mm,
       ROUND(LAG(amount_mm, 12) OVER (ORDER BY issue_month), 1)                 AS last_year_mm,
       ROUND(amount_mm / LAG(amount_mm, 12) OVER (ORDER BY issue_month) - 1, 4) AS yoy_change
FROM monthly
ORDER BY issue_month
""").df()

,issue_month,amount_mm,last_year_mm,yoy_change
0,2015-01-01,533.1,NaN,NaN
1,2015-02-01,366.9,NaN,NaN
2,2015-03-01,390.0,NaN,NaN
3,2015-04-01,539.4,NaN,NaN
4,2015-05-01,483.2,NaN,NaN
5,2015-06-01,429.8,NaN,NaN
6,2015-07-01,696.2,NaN,NaN
7,2015-08-01,555.3,NaN,NaN
8,2015-09-01,450.2,NaN,NaN
9,2015-10-01,738.2,NaN,NaN


**Findings**
* **Apr-2016:** −43.4% MoM, but only **−0.4% YoY**. April was normal; the MoM drop is a **base effect** from the March spike (base effect ≠ seasonality: the spike was a one-off, not a yearly pattern).
* **Jul-2016 to Apr-2017:** negative YoY for **10 consecutive months**. This is the real slowdown, and it recovers from May 2017.
* YoY can also be fooled: Mar-2017 is −42% YoY only because Mar-2016 was the spike.

## 3. Decomposing the slowdown
`funded volume = applications × approval rate × average loan size`, compared over two **equal-length** windows (H2-2015 vs. H2-2016).

In [4]:
decomp = con.sql("""
WITH a AS (                                   -- monthly approvals and funded amount
    SELECT DATE_TRUNC('month', issue_date) AS month,
           COUNT(*)                        AS n_accepted,
           SUM(funded_amnt)                AS funded
    FROM loans_clean
    GROUP BY month
),
r AS (                                        -- monthly rejections (by application date)
    SELECT DATE_TRUNC('month', "Application Date") AS month,
           COUNT(*)                                AS n_rejected
    FROM rejected
    GROUP BY month
),
m AS (                                        -- join and label the two comparison windows
    SELECT a.month, a.n_accepted, r.n_rejected, a.funded,
           CASE WHEN a.month BETWEEN '2015-07-01' AND '2015-12-01' THEN '2015 H2'
                WHEN a.month BETWEEN '2016-07-01' AND '2016-12-01' THEN '2016 H2'
           END AS period
    FROM a
    JOIN r ON a.month = r.month
)
SELECT period,
       SUM(n_accepted + n_rejected)                                   AS applications,
       ROUND(SUM(n_accepted) * 1.0 / SUM(n_accepted + n_rejected), 4) AS approval_rate,
       ROUND(SUM(funded) / SUM(n_accepted), 0)                        AS avg_loan_size,  -- weighted: sum first, then divide
       ROUND(SUM(funded) / 1000000, 1)                                AS funded_mm
FROM m
WHERE period IN ('2015 H2', '2016 H2')
GROUP BY period
ORDER BY period
""").df().set_index("period").T

decomp["change"] = decomp["2016 H2"] / decomp["2015 H2"] - 1
decomp

period,2015 H2,2016 H2,change
applications,1.871026e+06,2.879775e+06,0.539142
approval_rate,1.288000e-01,7.040000e-02,-0.453416
avg_loan_size,1.525000e+04,1.416100e+04,-0.071410
funded_mm,3.675200e+03,2.869900e+03,-0.219117


In [5]:
# Sanity check: component changes must multiply out to the total change
c = decomp["change"]
implied = (1 + c["applications"]) * (1 + c["approval_rate"]) * (1 + c["avg_loan_size"]) - 1
print(f"implied funded change: {implied:.1%} | actual: {c['funded_mm']:.1%}")

implied funded change: -21.9% | actual: -21.9%


**Bug found and fixed:** my first version labelled the whole of 2016 as "2016 H2", which compared 6 months against 12 and showed funded volume *rising* 74%. That contradicted the YoY result, which is how I caught it.

**Findings:** applications **+53.9%**, approval rate **−45.4%**, average loan size −7.1%, funded **−21.9%**. Demand was not the problem; the drop came from approvals. The data does not yet say *why*. Next, I need to separate **policy tightening**, an **applicant mix shift** (weaker applicants in the surge; compare `Risk_Score` bands), and a **funding constraint** (investor pullback after May 2016).

## 4. DTI: profile first, then bin (pandas)

In [6]:
dti = con.sql("""
SELECT dti
FROM cohort
""").df()

print("missing values:", dti["dti"].isna().sum())
print("DTI > 40:", (dti["dti"] > 40).sum())
dti["dti"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99])

missing values: 2
DTI > 40: 34


count    445741.000000
mean         18.139019
std           8.599754
min           0.000000
1%            2.180000
25%          11.760000
50%          17.570000
75%          24.080000
99%          37.800000
max         999.000000
Name: dti, dtype: float64

Max = **999** while the 99th percentile is 37.8: a DTI of 999 means debt payments are ~10× income, so it is almost certainly a placeholder or data error. The median (17.6) is a better "typical" value than the mean (18.1), which outliers pull upward.

In [7]:
df = con.sql("""
SELECT dti,
       CASE WHEN loan_status IN ('Charged Off', 'Late (31-120 days)') THEN 1
            WHEN loan_status = 'Fully Paid' THEN 0 END AS bad_flag
FROM cohort
""").df()

# Left-closed bins: [0, 10), [10, 20), ...; inf catches the 999 outliers
df["dti_band"] = pd.cut(df["dti"], bins=[0, 10, 20, 30, 40, float("inf")], right=False)

result = df.groupby("dti_band", observed=True, dropna=False).agg(   # dropna=False keeps the 2 missing rows
    n_loans=("bad_flag", "size"),
    bad_rate=("bad_flag", "mean"),     # mean() skips NULL (indeterminate) loans
)
print("total loans in bins:", result["n_loans"].sum())
result

total loans in bins: 445743


,n_loans,bad_rate
dti_band,,
"[0.0, 10.0)",81307,0.111551
"[10.0, 20.0)",187092,0.129069
"[20.0, 30.0)",135375,0.167242
"[30.0, 40.0)",41933,0.206383
"[40.0, inf)",34,0.235294
NaN,2,0.5


Bad rate rises **monotonically** (11.2% → 20.6%). The `40+` (34 loans) and missing (2 loans) bins are too small to trust and would be merged in a scorecard.

## 5. Employment length

In [8]:
df = con.sql("""
SELECT emp_length,
       CASE WHEN loan_status IN ('Charged Off', 'Late (31-120 days)') THEN 1
            WHEN loan_status = 'Fully Paid' THEN 0 END AS bad_flag
FROM cohort
""").df()

# Text sorts '10+ years' between '1 year' and '2 years', so map to numbers for ordering
emp_map = {"< 1 year": 0, "1 year": 1, "2 years": 2, "3 years": 3, "4 years": 4, "5 years": 5,
           "6 years": 6, "7 years": 7, "8 years": 8, "9 years": 9, "10+ years": 10}
df["emp_years"] = df["emp_length"].map(emp_map)

emp = (df.groupby("emp_years", dropna=False)
         .agg(n_loans=("bad_flag", "size"), bad_rate=("bad_flag", "mean"))
         .sort_index(na_position="first"))
emp

,n_loans,bad_rate
emp_years,,
NaN,28616,0.211115
0.0,37147,0.151031
1.0,29811,0.151666
2.0,40558,0.144104
3.0,36216,0.144668
4.0,26426,0.145339
5.0,26047,0.145436
6.0,19387,0.14145
7.0,21109,0.140576


**Missing employment length is the strongest signal (21.1%)**, likely unemployed, self-employed or retired borrowers. This is missing-not-at-random, so it keeps its own bin instead of being imputed. Other tenures sit at 14–15%, except `10+ years` (12.9%).

## 6. Loan purpose

In [9]:
df = con.sql("""
SELECT purpose,
       CASE WHEN loan_status IN ('Charged Off', 'Late (31-120 days)') THEN 1
            WHEN loan_status = 'Fully Paid' THEN 0 END AS bad_flag
FROM cohort
""").df()

purpose = df.groupby("purpose", dropna=False).agg(
    n_loans=("bad_flag", "size"),
    bad_rate=("bad_flag", "mean"),
)
purpose["pct_loans"] = purpose["n_loans"] / purpose["n_loans"].sum()
purpose = purpose.sort_values("n_loans", ascending=False)
purpose

,n_loans,bad_rate,pct_loans
purpose,,,
debt_consolidation,255890,0.152336,0.574075
credit_card,110780,0.119993,0.248529
home_improvement,25822,0.129827,0.05793
other,23488,0.163189,0.052694
major_purchase,8237,0.141608,0.018479
medical,5032,0.175313,0.011289
car,4285,0.126751,0.009613
small_business,4062,0.228269,0.009113
moving,3195,0.206886,0.007168


Debt consolidation is 57% of loans. Credit card refinancing is safest (12.0%), small business riskiest (22.8%). `wedding` (11) and `educational` (1) show 0% only by chance; tiny categories must be **merged** (e.g. into `other`), never kept as their own bin.

## 7. Ranking the variables
Simple separation measure: highest ÷ lowest bin bad rate, ignoring bins with fewer than 300 loans.

In [10]:
ranking = pd.DataFrame({
    "variable":     ["grade", "annual_inc", "home_ownership", "dti", "emp_length", "purpose"],
    "min_bad_rate": [0.0542, 0.1033, 0.1176, 0.1116, 0.1292, 0.1200],
    "max_bad_rate": [0.4198, 0.1961, 0.1718, 0.2064, 0.2111, 0.2283],
})

ranking["ratio"] = ranking["max_bad_rate"] / ranking["min_bad_rate"]
ranking = ranking.sort_values("ratio", ascending=False)
ranking

,variable,min_bad_rate,max_bad_rate,ratio
0,grade,0.0542,0.4198,7.745387
5,purpose,0.1200,0.2283,1.902500
1,annual_inc,0.1033,0.1961,1.898354
3,dti,0.1116,0.2064,1.849462
4,emp_length,0.1292,0.2111,1.633901
2,home_ownership,0.1176,0.1718,1.460884


**Findings**
* Grade (7.8×) separates risk about **4×** better than any single application variable (1.5–1.9×), because it is the output of Lending Club's model and already combines many variables. It is the **benchmark** for a new model, not an input.
* To approach grade, several variables must be **combined** in a multivariate model.
* The max/min ratio only looks at two extreme bins and ignores bin sizes, so purpose is overrated (`small_business` is just 0.9% of loans). **Information Value** is the proper measure (next).